In [1]:
# ================================================
# Notebook 02 - Build Respiratory Model
# ================================================

# Install packages (only needed in Colab)
%pip install scanpy anndata pandas numpy tqdm requests --quiet

# Handle Colab-specific imports
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully")
except ImportError:
    print("Running outside Colab - skipping Drive mount")
except Exception as e:
    print(f"Drive mount failed: {e}")

import scanpy as sc
import os
from pathlib import Path
from tqdm import tqdm
import requests

print("✅ Environment ready!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully
✅ Environment ready!


In [2]:
import os
from pathlib import Path
from tqdm import tqdm
import requests

# Define correct paths in Drive
base_path = Path("/content/drive/MyDrive/HEALTH")
data_path = base_path / "data"
data_path.mkdir(parents=True, exist_ok=True)

print("Data folder ready at:", data_path)

# Download HLCA Full
url = "https://datasets.cellxgene.cziscience.com/dbb5ad81-1713-4aee-8257-396fbabe7c6e.h5ad"
save_path = data_path / "hlca_full.h5ad"

if not save_path.exists():
    print("Downloading HLCA Full (20.36 GB)... This will take some time.")
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(save_path, "wb") as f, tqdm(
        desc="Downloading HLCA Full", 
        total=total_size, 
        unit='iB', 
        unit_scale=True, 
        unit_divisor=1024
    ) as bar:
        for data in response.iter_content(chunk_size=1024*1024):
            size = f.write(data)
            bar.update(size)
    print("✅ HLCA Full downloaded successfully!")
else:
    print("✅ HLCA Full already exists.")

print(f"File location: {save_path}")
print(f"Size: {save_path.stat().st_size / (1024**3):.2f} GB")

Data folder ready at: /content/drive/MyDrive/HEALTH/data
✅ HLCA Full already exists.
File location: /content/drive/MyDrive/HEALTH/data/hlca_full.h5ad
Size: 20.36 GB


In [3]:
import os
from pathlib import Path

print("=== Checking Colab temporary storage ===")
colab_temp = Path("/content")
h5ad_in_temp = list(colab_temp.rglob("*.h5ad"))

if h5ad_in_temp:
    print(f"Found {len(h5ad_in_temp)} .h5ad files in Colab temp:")
    for f in h5ad_in_temp:
        print(f"  - {f}")
else:
    print("No .h5ad files in Colab temp storage.")

print("\n=== Checking if Drive is mounted ===")
drive_path = Path("/content/drive/MyDrive")
if drive_path.exists():
    print("Drive is mounted.")
    print("Contents of MyDrive/HEALTH:")
    health_path = drive_path / "HEALTH"
    if health_path.exists():
        for item in health_path.iterdir():
            print(f"  - {item.name}")
    else:
        print("HEALTH folder not found in MyDrive.")
else:
    print("Drive is NOT mounted. Please mount it.")

print("\nIf you see files in Colab temp, tell me and I'll give you code to move them to Drive.")

=== Checking Colab temporary storage ===
Found 1 .h5ad files in Colab temp:
  - /content/drive/MyDrive/HEALTH/data/hlca_full.h5ad

=== Checking if Drive is mounted ===
Drive is mounted.
Contents of MyDrive/HEALTH:
  - models
  - data

If you see files in Colab temp, tell me and I'll give you code to move them to Drive.


In [4]:
from pathlib import Path

print("Searching entire Drive for all .h5ad files...\n")

found_files = list(Path("/content/drive/MyDrive").rglob("*.h5ad"))

if found_files:
    print(f"Found {len(found_files)} .h5ad files:\n")
    for f in sorted(found_files):
        print(f"  - {f}")
else:
    print("No .h5ad files found anywhere in MyDrive.")

print("\nIf files were found above, tell me their paths and I'll give you code to move them into the correct data folder.")

Searching entire Drive for all .h5ad files...

Found 1 .h5ad files:

  - /content/drive/MyDrive/HEALTH/data/hlca_full.h5ad

If files were found above, tell me their paths and I'll give you code to move them into the correct data folder.


In [5]:
import os
from pathlib import Path

# Define the correct paths
BASE_PATH = Path("/content/drive/MyDrive/HEALTH")
DATA_PATH = BASE_PATH / "data"
MODELS_PATH = BASE_PATH / "models"

# Create folders if they don't exist
DATA_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

print("✅ Folders created/verified:")
print(f"Data folder   : {DATA_PATH}")
print(f"Models folder : {MODELS_PATH}")

# List what currently exists
print("\nCurrent contents of HEALTH folder:")
for item in BASE_PATH.iterdir():
    print(f"  - {item.name} ({'folder' if item.is_dir() else 'file'})")

# Now move your existing .h5ad files into the data folder if they are elsewhere
print("\nLooking for .h5ad files to move into data folder...")
for file in BASE_PATH.rglob("*.h5ad"):
    if file.parent != DATA_PATH:
        new_location = DATA_PATH / file.name
        if not new_location.exists():
            file.rename(new_location)
            print(f"Moved: {file.name}")
        else:
            print(f"Already exists: {file.name}")

✅ Folders created/verified:
Data folder   : /content/drive/MyDrive/HEALTH/data
Models folder : /content/drive/MyDrive/HEALTH/models

Current contents of HEALTH folder:
  - models (folder)
  - data (folder)

Looking for .h5ad files to move into data folder...


In [6]:
import os
from pathlib import Path

# Print current working directory and Drive contents
print("Current working directory:", os.getcwd())

drive_path = Path("/content/drive/MyDrive/HEALTH")
print("\nContents of HEALTH folder:")
if drive_path.exists():
    for item in drive_path.iterdir():
        print(f"  - {item.name} ({'folder' if item.is_dir() else 'file'})")
else:
    print("HEALTH folder not found!")

# List data folder contents
data_path = drive_path / "data"
print("\nContents of data folder:")
if data_path.exists():
    for item in data_path.iterdir():
        print(f"  - {item.name}")
else:
    print("data folder not found!")

Current working directory: /content

Contents of HEALTH folder:
  - models (folder)
  - data (folder)

Contents of data folder:
  - hlca_full.h5ad


# Notebook 02 — Build the Healthy Respiratory Model

**Purpose**: Distil 1.2 million healthy adult lung cells from the HLCA Full dataset — cross-referenced with fetal lung development data and spatial Visium tissue maps — into a compact, reusable model artefact that the HEALTH FastAPI service can load in milliseconds.

**What this notebook produces** (`models/respiratory_model/`):

| File | Biological meaning | What it stores |
|------|--------------------|----------------|
| `meta.json` | Build provenance | Date, cell count, donor count, gene count, all build parameters |
| `cell_type_composition.json` | The cellular architecture of a healthy lung | Expected fraction of each cell type ± std across donors |
| `gene_stats.json` | The molecular state of healthy lung tissue | Mean, std, percentiles for every expressed gene |
| `cell_type_profiles.json` | What each cell type is actually doing | Per-cell-type expression of its top marker genes |
| `pathway_baselines.json` | The background activity of key biological processes | Baseline expression of 12 pathway gene sets |
| `fetal_reference.json` | The developmental context | Genes that distinguish fetal from adult lung — flags dedifferentiation |
| `spatial_baselines.json` | Where in the lung genes are normally active | Per-anatomical-region gene statistics from Visium tissue sections |

**What this notebook does NOT do**: disease detection, anomaly scoring, or anything clinical. Every computation here defines what is *normal*.

---

## Why build a healthy baseline?

The HEALTH platform's intelligence engine works by comparison. To recognise that something is wrong, we first need a precise, data-derived portrait of what is right — not a textbook description, but a statistical model built from the actual molecular state of healthy human lung tissue across diverse donors, developmental stages, and anatomical regions.

This notebook builds that portrait across four dimensions:

1. **Cellular composition** — the expected proportions of each cell type in a healthy adult lung, derived from ~80 donors
2. **Gene expression ranges** — the typical expression of each gene, including its full distribution (not just the mean)
3. **Pathway activity** — the background tone of 12 major biological processes in healthy tissue
4. **Spatial and developmental context** — where in the lung genes are normally active, and which patterns belong to development vs. adult health

Any future patient sample can be measured against all four dimensions simultaneously.

---
## Section 1 — Environment Setup

Install the libraries we need. These are identical to the HEALTH FastAPI service dependencies, so any processing we do here will match exactly what the service produces.

In [7]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

try:
    import anndata, scanpy
    print(f'scanpy {scanpy.__version__}, anndata {anndata.__version__} — ready.')
except ImportError:
    pip_install('scanpy>=1.10.0', 'anndata>=0.10.0')
    print('Installed.')

scanpy 1.12.1, anndata 0.12.11 — ready.


/tmp/ipykernel_28714/727354933.py:8: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f'scanpy {scanpy.__version__}, anndata {anndata.__version__} — ready.')
/tmp/ipykernel_28714/727354933.py:8: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(f'scanpy {scanpy.__version__}, anndata {anndata.__version__} — ready.')


In [8]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import issparse

print('All imports successful.')
print(f'numpy {np.__version__} | pandas {pd.__version__} | scanpy {sc.__version__} | anndata {ad.__version__}')


def normalize_cp10k_log1p(X_raw):
    """Normalize raw counts to log1p(CP10K)."""
    totals = X_raw.sum(axis=1, keepdims=True)
    totals = np.where(totals == 0, 1, totals)
    return np.log1p(X_raw / totals * 10_000)


All imports successful.
numpy 2.0.2 | pandas 2.3.3 | scanpy 1.12.1 | anndata 0.12.11


/tmp/ipykernel_28714/4183817469.py:13: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f'numpy {np.__version__} | pandas {pd.__version__} | scanpy {sc.__version__} | anndata {ad.__version__}')
/tmp/ipykernel_28714/4183817469.py:13: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(f'numpy {np.__version__} | pandas {pd.__version__} | scanpy {sc.__version__} | anndata {ad.__version__}')


---
## Section 2 — Mount Google Drive and Set Paths

All large data files live in Google Drive. The model output artefacts (a few MB total) are saved back to Drive, then downloaded to the HEALTH repo.

**Edit the paths below if your Drive layout is different.**

In [9]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [10]:
# ── Edit these paths to match your Drive layout ────────────────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive')
DATA_DIR      = DRIVE_ROOT / 'HEALTH' / 'data'
MODEL_OUT_DIR = DRIVE_ROOT / 'HEALTH' / 'models' / 'respiratory_model'
# ──────────────────────────────────────────────────────────────────────────────

# Primary HLCA Full — 2.28M cells, 55K genes (20 GB)
HLCA_FULL_PATH = DATA_DIR / 'hlca_full.h5ad'

# Fetal lung — 71K cells, 26K genes — developmental context
FETAL_PATH = DATA_DIR / 'lung_fetal_assembled_10domains.h5ad'

# Spatial Visium — 4,992 spots each — anatomical context
VISIUM_BRONCHUS_PATH    = DATA_DIR / 'lung_tissues_visium_bronchus.h5ad'
VISIUM_PARENCHYMA_PATH  = DATA_DIR / 'lung_tissues_visium_parenchyma.h5ad'

MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Report which files are present
for label, path in [
    ('HLCA Full',          HLCA_FULL_PATH),
    ('Fetal lung',         FETAL_PATH),
    ('Visium bronchus',    VISIUM_BRONCHUS_PATH),
    ('Visium parenchyma',  VISIUM_PARENCHYMA_PATH),
]:
    status = 'found' if path.exists() else 'MISSING'
    print(f'  {label:<22} {status}  ({path.name})')

if not HLCA_FULL_PATH.exists():
    raise FileNotFoundError(f'HLCA Full not found at {HLCA_FULL_PATH}')

  HLCA Full              found  (hlca_full.h5ad)
  Fetal lung             MISSING  (lung_fetal_assembled_10domains.h5ad)
  Visium bronchus        MISSING  (lung_tissues_visium_bronchus.h5ad)
  Visium parenchyma      MISSING  (lung_tissues_visium_parenchyma.h5ad)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ⚡  KERNEL RESTART?  Run ONLY this cell, then continue from Cell 14   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import json, time
from datetime import datetime, timezone
from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import issparse
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# ── Paths (keep in sync with the config cell above) ─────────────────────────────
DRIVE_ROOT             = Path('/content/drive/MyDrive')
DATA_DIR               = DRIVE_ROOT / 'HEALTH' / 'data'
MODEL_OUT_DIR          = DRIVE_ROOT / 'HEALTH' / 'models' / 'respiratory_model'
HLCA_FULL_PATH         = DATA_DIR / 'hlca_full.h5ad'
FETAL_PATH             = DATA_DIR / 'lung_fetal_assembled_10domains.h5ad'
VISIUM_BRONCHUS_PATH   = DATA_DIR / 'lung_tissues_visium_bronchus.h5ad'
VISIUM_PARENCHYMA_PATH = DATA_DIR / 'lung_tissues_visium_parenchyma.h5ad'
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Intermediate artefact paths ────────────────────────────────────────────
_OUT_GENE_STATS  = MODEL_OUT_DIR / 'gene_stats.json'
_OUT_COMPOSITION = MODEL_OUT_DIR / 'cell_type_composition.json'
_OUT_PROFILES    = MODEL_OUT_DIR / 'cell_type_profiles.json'
_OUT_CT_GENES    = MODEL_OUT_DIR / 'gene_stats_by_cell_type.json'
_OUT_PATHWAYS    = MODEL_OUT_DIR / 'pathway_baselines.json'

# ── Utility functions ──────────────────────────────────────────────────────
def normalize_cp10k_log1p(X_raw):
    totals = X_raw.sum(axis=1, keepdims=True)
    totals = np.where(totals == 0, 1, totals)
    return np.log1p(X_raw / totals * 10_000)

def save_json(data, path, label=''):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2, allow_nan=False)
    kb = Path(path).stat().st_size / 1024
    print(f'  ✓ saved {label or Path(path).name}  ({kb:.0f} KB)')

def load_json(path):
    with open(path) as f:
        return json.load(f)

def infer_compartment(cell_type):
    ct = cell_type.lower()
    if any(k in ct for k in ['at1', 'at2', 'type 1', 'type 2', 'alveolar type', 'alveolar macrophage']):
        return 'alveolar'
    if any(k in ct for k in ['t cell', 'b cell', 'nk ', 'dendritic', 'mast', 'plasma',
                              'monocyte', 'macrophage', 'neutrophil', 'eosinophil', 'basophil']):
        return 'immune'
    if any(k in ct for k in ['club', 'goblet', 'ciliated', 'basal', 'secretory', 'epithelial', 'airway']):
        return 'airway_epithelial'
    if any(k in ct for k in ['endothelial', 'capillary', 'venous', 'arterial', 'lymphatic']):
        return 'endothelial'
    if any(k in ct for k in ['fibroblast', 'smooth muscle', 'pericyte', 'mesothelial']):
        return 'stromal'
    return 'other'

print('✓ Kernel recovery complete — imports, paths, and helpers restored.')
_e = lambda p: '✓' if Path(p).exists() else '✗'
print(f'  HLCA Full:         {_e(HLCA_FULL_PATH)}')
print(f'  Fetal lung:        {_e(FETAL_PATH)}')
print(f'  Visium bronchus:   {_e(VISIUM_BRONCHUS_PATH)}')
print(f'  Visium parenchyma: {_e(VISIUM_PARENCHYMA_PATH)}')
print(f'  gene_stats.json:   {_e(_OUT_GENE_STATS)}')
print(f'  CT profiles:       {_e(_OUT_PROFILES)}')
print(f'  CT gene stats:     {_e(_OUT_CT_GENES)}')
print(f'  pathway_baselines: {_e(_OUT_PATHWAYS)}')
print()
print('→ Re-run from Cell 14 (Load HLCA Full) — saved sections load in <2s.')

---
## Section 3 — Load HLCA Full

The HLCA Full is 20 GB. We open it with `backed='r'` — this keeps the expression matrix (`.X`) on disk and streams only the slices we request. The cell metadata (`.obs`) loads into RAM; at ~300 MB it is manageable.

**Rule**: never call `adata.X` or `adata.to_memory()` without slicing first. Always use integer positional indices and sort them before slicing to minimise disk seeks.

In [11]:
print('Loading HLCA Full with backed="r"...')
t0 = time.time()
adata = ad.read_h5ad(HLCA_FULL_PATH, backed='r')
print(f'Loaded in {time.time() - t0:.1f}s')
print(f'Shape: {adata.n_obs:,} cells × {adata.n_vars:,} genes')

Loading HLCA Full with backed="r"...
Loaded in 130.9s
Shape: 2,282,447 cells × 55,329 genes


In [ ]:
# Detect whether var_names are gene symbols or Ensembl IDs.
# CellxGene h5ad files sometimes use Ensembl IDs as var_names and store
# the human-readable symbol in adata.var['feature_name'].
# _GENE_NAMES is the authoritative gene symbol list used by every artefact.

_var_sample = str(adata.var_names[0])
_using_ensembl = _var_sample.startswith('ENSG') or _var_sample.startswith('ENSMUSG')

if _using_ensembl:
    # Try common CellxGene symbol columns in order of preference
    _symbol_cols = ['feature_name', 'gene_name', 'gene_symbols', 'symbol']
    _sym_col = next((c for c in _symbol_cols if c in adata.var.columns), None)
    if _sym_col:
        _GENE_NAMES = list(adata.var[_sym_col].astype(str))
        print(f'var_names are Ensembl IDs. Using adata.var["{_sym_col}"] as gene symbols.')
    else:
        # No symbol column — fall back to var_names and warn loudly
        _GENE_NAMES = list(adata.var_names)
        print('WARNING: var_names are Ensembl IDs but no symbol column found.')
        print('Available var columns:', list(adata.var.columns))
        print('gene_stats.json will use Ensembl IDs — anomaly detector will not match gene symbols.')
else:
    _GENE_NAMES = list(adata.var_names)
    print(f'var_names appear to be gene symbols (e.g. {_var_sample!r}). Using directly.')

print(f'Total genes: {len(_GENE_NAMES):,}')
print(f'Sample names: {_GENE_NAMES[:5]}')


In [12]:
# Confirm the key metadata columns we need
for col in ['disease', 'ann_finest_level', 'donor_id']:
    assert col in adata.obs.columns, f'Missing required column: {col}'

print('All required columns present.\n')
print('All obs columns:')
for col in adata.obs.columns:
    print(f'  {col:<35} n_unique={adata.obs[col].nunique()}')

All required columns present.

All obs columns:
  suspension_type                     n_unique=2
  donor_id                            n_unique=484
  is_primary_data                     n_unique=2
  assay_ontology_term_id              n_unique=8
  cell_type_ontology_term_id          n_unique=51
  development_stage_ontology_term_id  n_unique=80
  disease_ontology_term_id            n_unique=16
  self_reported_ethnicity_ontology_term_id n_unique=5
  tissue_ontology_term_id             n_unique=4
  sex_ontology_term_id                n_unique=3
  3'_or_5'                            n_unique=2
  BMI                                 n_unique=77
  age_or_mean_of_age_range            n_unique=97
  age_range                           n_unique=7
  anatomical_region_ccf_score         n_unique=7
  ann_coarse_for_GWAS_and_modeling    n_unique=39
  ann_finest_level                    n_unique=62
  ann_level_1                         n_unique=5
  ann_level_2                         n_unique=12
  ann_

In [13]:
# Disease distribution — we will use only 'normal'
disease_counts = adata.obs['disease'].value_counts()
print('Disease distribution:')
print(disease_counts.to_string())
print(f'\nNormal cells: {disease_counts.get("normal", 0):,}')

Disease distribution:
disease
normal                                   1305099
COVID-19                                  341761
pulmonary fibrosis                        268932
interstitial lung disease                  68456
chronic obstructive pulmonary disease      67943
lung adenocarcinoma                        62807
pneumonia                                  31923
chronic rhinitis                           29137
lung large cell carcinoma                  21167
squamous cell lung carcinoma               20631
cystic fibrosis                            17590
lymphangioleiomyomatosis                   12374
pleomorphic carcinoma                      10765
hypersensitivity pneumonitis               10379
non-specific interstitial pneumonia         8597
pulmonary sarcoidosis                       4886

Normal cells: 1,305,099


---
## Section 4 — Isolate Healthy Adult Cells

We keep only cells labelled `disease == 'normal'`. These came from donors with no known lung pathology at the time of biopsy. This is our ground truth for adult healthy lung biology.

Note that 'normal' here means *donor-level* normality — not every individual cell is guaranteed to be perfectly healthy, but the population-level statistics reflect what a functioning lung looks like.

In [14]:
# Work with obs (DataFrame) — never load .X for the whole corpus
healthy_obs = adata.obs[adata.obs['disease'] == 'normal'].copy()

n_healthy    = len(healthy_obs)
n_donors     = healthy_obs['donor_id'].nunique()
n_cell_types = healthy_obs['ann_finest_level'].nunique()

print(f'Healthy cells:      {n_healthy:,}')
print(f'Healthy donors:     {n_donors}')
print(f'Cell types present: {n_cell_types}')
print(f'Healthy fraction:   {n_healthy / adata.n_obs:.1%}')

Healthy cells:      1,305,099
Healthy donors:     268
Cell types present: 62
Healthy fraction:   57.2%


In [15]:
ct_counts = healthy_obs['ann_finest_level'].value_counts()
print(f'Top 20 cell types in healthy lung (of {len(ct_counts)} total):')
print(ct_counts.head(20).to_string())

Top 20 cell types in healthy lung (of 62 total):
ann_finest_level
Unknown                      213108
Alveolar macrophages         169539
AT2                          117546
Multiciliated (non-nasal)     67128
Goblet (nasal)                65787
CD8 T cells                   62542
Suprabasal                    54007
Basal resting                 49020
CD4 T cells                   43698
Monocyte-derived Mph          42608
Classical monocytes           37768
Club (nasal)                  34375
NK cells                      33859
EC general capillary          33752
Alveolar fibroblasts          27172
AT1                           24565
B cells                       19332
Adventitial fibroblasts       17810
Non-classical monocytes       15223
DC2                           14807


---
## Section 5 — Cell Type Composition (Artefact 1)

### What this artefact captures biologically

A healthy lung is not a random mixture of cells. It maintains a precise architectural organisation:
- **Alveolar macrophages** dominate the air-facing surface (~25%), surveilling the barrier
- **AT2 cells** maintain surfactant production (~8–12%), keeping alveoli open
- **Ciliated cells** sweep debris upward (~5–8%), providing mucociliary clearance
- **Fibroblasts** provide structural scaffold; **endothelial cells** line the vasculature

When any of these proportions shift — macrophages expand during infection, AT2 cells deplete in fibrosis — it is a measurable biological signal. This artefact encodes those expected proportions so the Abnormality Detector can compute a Z-score for any submitted sample.

### Why compute fractions per donor before averaging?

A donor contributing 50,000 cells would dominate a naive cross-cell average. By computing each donor's cell type fractions independently, then averaging, every person contributes equally regardless of how many cells were sequenced from their sample. This removes technical noise and preserves biological signal.

In [16]:
def infer_compartment(cell_type: str) -> str:
    """Assign a broad biological compartment from a cell type name."""
    ct = cell_type.lower()
    if any(k in ct for k in ['at1', 'at2', 'type 1', 'type 2', 'alveolar type']):
        return 'alveolar'
    if any(k in ct for k in ['alveolar macrophage']):
        return 'alveolar'  # resident macrophages belong to the alveolar niche
    if any(k in ct for k in ['t cell', 'b cell', 'nk ', 'dendritic', 'mast', 'plasma',
                              'monocyte', 'macrophage', 'neutrophil', 'eosinophil', 'basophil']):
        return 'immune'
    if any(k in ct for k in ['basal', 'ciliated', 'secretory', 'airway', 'tracheal',
                              'bronchial', 'goblet', 'club', 'serous', 'submucosal']):
        return 'airway'
    if any(k in ct for k in ['fibroblast', 'smooth muscle', 'pericyte', 'mesothelial']):
        return 'stromal'
    if any(k in ct for k in ['endothelial', 'capillary', 'venous', 'arterial', 'lymphatic']):
        return 'vascular'
    if any(k in ct for k in ['tuft', 'ionocyte', 'neuroendocrine', 'nerve']):
        return 'rare_specialized'
    return 'unknown'


In [17]:
def compute_cell_type_composition(healthy_obs: pd.DataFrame) -> dict:
    # Step 1: fraction of each cell type within each donor
    donor_fractions = (
        healthy_obs
        .groupby('donor_id')['ann_finest_level']
        .value_counts(normalize=True)   # fraction within each donor
        .unstack(fill_value=0.0)        # donors × cell_types matrix
    )
    # Step 2: aggregate across donors (each donor weighted equally)
    mean_f = donor_fractions.mean(axis=0)
    std_f  = donor_fractions.std(axis=0)
    n_donors_with_type = (donor_fractions > 0).sum(axis=0)

    return {
        ct: {
            'mean_fraction': float(mean_f[ct]),
            'std_fraction':  float(std_f[ct]),
            'n_donors':      int(n_donors_with_type[ct]),
            'compartment':   infer_compartment(ct),
        }
        for ct in mean_f.index
    }


print('Computing cell type composition...')
t0 = time.time()
composition = compute_cell_type_composition(healthy_obs)
print(f'Done in {time.time() - t0:.1f}s. {len(composition)} cell types.')

Computing cell type composition...
Done in 0.1s. 62 cell types.


/tmp/ipykernel_28714/256639399.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('donor_id')['ann_finest_level']


In [18]:
comp_df = pd.DataFrame(composition).T.sort_values('mean_fraction', ascending=False)
print('Top 20 cell types by mean fraction in healthy lung:')
print(comp_df[['mean_fraction', 'std_fraction', 'n_donors', 'compartment']].head(20).to_string(float_format='{:.4f}'.format))
print(f'\nCompartment breakdown:')
print(comp_df['compartment'].value_counts().to_string())
print(f'\nFraction sum: {comp_df["mean_fraction"].sum():.4f}  (should be ≈1.0)')

Top 20 cell types by mean fraction in healthy lung:
                          mean_fraction std_fraction n_donors compartment
Unknown                          0.1101       0.1760      162     unknown
Alveolar macrophages             0.0600       0.1522      189    alveolar
AT2                              0.0475       0.1155      170    alveolar
Multiciliated (non-nasal)        0.0375       0.0858      253      airway
Suprabasal                       0.0341       0.1093      169      airway
Goblet (nasal)                   0.0289       0.0950      174      airway
Monocyte-derived Mph             0.0255       0.0869      190      immune
Basal resting                    0.0240       0.0715      192      airway
CD8 T cells                      0.0237       0.0560      214      immune
Classical monocytes              0.0170       0.0441      192      immune
CD4 T cells                      0.0162       0.0435      207      immune
Club (nasal)                     0.0136       0.0504      10

---
## Section 6 — Global Gene Statistics (Artefact 2)

### What this artefact captures biologically

Every gene in the lung has a characteristic expression level in health. Some are constitutively expressed across all cells (housekeeping genes like *GAPDH*, *B2M*). Some are cell-type-restricted — *SFTPC* is almost exclusively produced by AT2 cells. Some are near-silent unless activated — *MX1* is an interferon-stimulated gene that sits at background in healthy tissue but spikes during viral infection.

This artefact captures the **full distribution** of each gene's expression across the healthy cell population. It is the most foundational piece of the model: every downstream analysis (pathway scoring, profile comparison, Z-score computation) ultimately refers back to these numbers.

### Why store percentiles rather than just mean ± std?

Gene expression is not Gaussian — it is highly skewed. Most cells express a gene at low or zero levels; a small number express it very highly. The standard deviation is pulled upward by these outliers and gives a misleading picture of "normal range."

Percentiles are **distribution-free** and robust to skew:
- **p5 / p95** define the extreme-but-still-healthy range. A sample value above p95 is definitively high for a healthy cell.
- **p25 / p75** define the interquartile range (IQR) — what most healthy cells look like.
- A sample expression value at p97 says more than "mean + 2σ" for a skewed gene.

The Abnormality Detector will use these percentile boundaries to compute percentile-rank scores that are not distorted by distributional assumptions.

### Why sample 15,000 cells?

Statistics from 15,000 cells are nearly identical to statistics from 1.2 million for this purpose. The mean stabilises within a few thousand cells; the 5th and 95th percentiles stabilise within ~10,000. Sampling lets this section run in under a minute instead of hours.

In [ ]:
GLOBAL_SAMPLE_SIZE  = 5_000   # cells to sample for global gene stats (reduced for Colab RAM)
MIN_EXPRESSING_FRAC = 0.01    # exclude genes expressed in <1% of healthy cells

rng = np.random.default_rng(seed=42)
# np.where returns integer positions — required for backed='r' slicing
healthy_indices   = np.where(adata.obs['disease'] == 'normal')[0]
n_sample          = min(GLOBAL_SAMPLE_SIZE, len(healthy_indices))
sampled_positions = np.sort(rng.choice(healthy_indices, size=n_sample, replace=False))

print(f'Healthy cells available: {len(healthy_indices):,}')
print(f'Sampling:                {n_sample:,} cells (seed=42)')
print('Reading from disk...')

t0 = time.time()
X_raw = adata.X[sampled_positions, :]
if issparse(X_raw):
    X_raw = X_raw.toarray()
print(f'Read {X_raw.shape[0]:,} × {X_raw.shape[1]:,} matrix in {time.time() - t0:.1f}s ({X_raw.nbytes / 1e9:.2f} GB)')

Healthy cells available: 1,305,099
Sampling:                15,000 cells (seed=42)
Reading from disk...
Read 15,000 × 55,329 matrix in 76.3s (3.32 GB)


: 

In [20]:
X_norm = normalize_cp10k_log1p(X_raw)
print(f'Normalized shape: {X_norm.shape}')
print(f'Value range: {X_norm.min():.3f} – {X_norm.max():.3f}')
print(f'Non-zero mean: {X_norm[X_norm > 0].mean():.3f}')

: 

: 

In [ ]:
def compute_gene_stats(X_norm: np.ndarray, gene_names: list, min_expressing_frac: float = 0.01) -> dict:
    """
    Compute per-gene statistics across healthy cells.

    Genes expressed in fewer than min_expressing_frac of cells are excluded:
    their statistics are too noisy to be useful as a reference.

    Threshold is 0.01 (1%) — captures disease-inducible genes (TNF, CXCL10, IFNG,
    SPP1, MMP9, MMP12, KRAS, MYC) that are near-silent in healthy tissue.

    Percentiles (p5, p25, p75, p95) are stored alongside mean/std because
    gene expression distributions are skewed — percentiles give a robust,
    assumption-free description of the healthy range.
    """
    pct_expressing = (X_norm > 0).mean(axis=0)
    keep_mask  = pct_expressing >= min_expressing_frac
    kept_genes = [g for g, k in zip(gene_names, keep_mask) if k]
    X_kept     = X_norm[:, keep_mask]
    pct_kept   = pct_expressing[keep_mask]

    print(f'Genes ≥{min_expressing_frac:.0%} expressing: {len(kept_genes):,} / {len(gene_names):,}')

    p5, p25, p75, p95 = np.percentile(X_kept, [5, 25, 75, 95], axis=0)
    means = X_kept.mean(axis=0)
    stds  = X_kept.std(axis=0)

    return {
        gene: {
            'mean':           float(means[i]),
            'std':            float(stds[i]),
            'p5':             float(p5[i]),
            'p25':            float(p25[i]),
            'p75':            float(p75[i]),
            'p95':            float(p95[i]),
            'pct_expressing': float(pct_kept[i]),
        }
        for i, gene in enumerate(kept_genes)
    }


gene_names = _GENE_NAMES
if _OUT_GENE_STATS.exists():
    gene_stats = load_json(_OUT_GENE_STATS)
    print(f'Loaded gene_stats from Drive: {len(gene_stats):,} genes (skipped recompute)')
else:
    print('Computing gene statistics...')
    t0 = time.time()
    gene_stats = compute_gene_stats(X_norm, gene_names, MIN_EXPRESSING_FRAC)
    print(f'Done in {time.time() - t0:.1f}s. {len(gene_stats):,} genes tracked.')
    save_json(gene_stats, _OUT_GENE_STATS, 'gene_stats.json')

In [ ]:
# Inspect landmark genes as a biological sanity check
# SFTPC/SFTPB: AT2 surfactant markers — should be present but cell-type-restricted
# MX1: interferon-stimulated — should be very low in healthy (not activated)
# MARCO: alveolar macrophage scavenger receptor — should be moderately expressed
# COL1A1: fibroblast collagen — low in healthy (no active fibrosis)
landmark_genes = ['SFTPC', 'SFTPB', 'MX1', 'MARCO', 'ACTA2', 'EPCAM', 'COL1A1', 'PECAM1']

print('Landmark gene statistics (log1p CP10K):')
print(f'{"Gene":<12} {"mean":>8} {"std":>8} {"p5":>8} {"p75":>8} {"p95":>8} {"pct_expr":>10}')
print('-' * 68)
for gene in landmark_genes:
    if gene in gene_stats:
        s = gene_stats[gene]
        print(f'{gene:<12} {s["mean"]:>8.3f} {s["std"]:>8.3f} {s["p5"]:>8.3f} {s["p75"]:>8.3f} {s["p95"]:>8.3f} {s["pct_expressing"]:>10.2%}')
    else:
        print(f'{gene:<12} not in filtered set (expressed in <5% of healthy cells)')

---
## Section 7 — Per-Cell-Type Expression Profiles (Artefact 3)

### What this artefact captures biologically

The global gene statistics above describe what the lung tissue expresses in aggregate. But a lung cell is not a generic cell — each type has a distinct molecular identity:

- **AT2 cells** produce *SFTPC*, *SFTPB*, *ABCA3* at high levels — surfactant components
- **Alveolar macrophages** express *MARCO*, *PPARG*, *C1QA* — scavenger and complement proteins
- **Ciliated cells** express *FOXJ1*, *DNAI1*, *CCNO* — ciliogenesis transcription factors
- **Fibroblasts** express *COL1A1*, *DCN*, *LUM* — extracellular matrix components

This artefact stores the **expected expression level of each cell type's top marker genes in healthy tissue**. This allows a finer question than "is this gene outside the normal range globally?" — it allows: **"is this cell type executing its normal molecular program?"**

A patient could have the right number of AT2 cells (composition is normal) but those cells are not producing SFTPC at the right level (profile is abnormal). That is a different and more subtle type of dysfunction that only this artefact can detect.

### Why store only the top 50 genes per cell type?

Storing all 55,000 genes for each of 70+ cell types would produce a ~100 MB JSON file. The top 50 most expressed genes per cell type are the most diagnostically meaningful — they represent what that cell type is actively *doing* in health. The long tail of lowly expressed genes adds noise rather than signal.

In [ ]:
CELLS_PER_TYPE     = 500
MIN_CELLS_FOR_TYPE = 100
TOP_MARKER_GENES   = 50
CT_GENE_MIN_PCT    = 0.10  # gene must be expressed in ≥10% of this cell type's cells
CT_GENE_MIN_SPAN   = 0.5   # ct_mean - global_mean must be ≥ this (enrichment filter)


def compute_cell_type_profiles(
    adata, healthy_obs: pd.DataFrame, gene_names: list, gene_stats: dict,
    cells_per_type: int = 500, min_cells: int = 100, top_n_genes: int = 50,
) -> tuple[dict, dict]:
    """
    For each cell type, sample up to cells_per_type healthy cells and compute:
    1. profiles — top N marker genes (used for cell-type composition scoring)
    2. ct_gene_stats — genes meaningfully enriched above global baseline
       (ct_mean - global_mean ≥ CT_GENE_MIN_SPAN), for CT-aware Z-scoring.

    Both are computed in the same disk-read loop — no extra reads needed.
    Returns (profiles, ct_gene_stats).
    """
    rng = np.random.default_rng(seed=42)
    profiles      = {}
    ct_gene_stats = {}

    for cell_type in sorted(healthy_obs['ann_finest_level'].unique()):
        mask      = (adata.obs['disease'] == 'normal') & (adata.obs['ann_finest_level'] == cell_type)
        positions = np.where(mask)[0]

        if len(positions) < min_cells:
            print(f'  skip  {cell_type} ({len(positions)} cells < {min_cells})')
            continue

        n_sample   = min(cells_per_type, len(positions))
        sample_pos = np.sort(rng.choice(positions, size=n_sample, replace=False))

        X_raw = adata.X[sample_pos, :]
        if issparse(X_raw):
            X_raw = X_raw.toarray()
        X_ct = normalize_cp10k_log1p(X_raw.astype(np.float32))
        del X_raw

        means    = X_ct.mean(axis=0)
        stds     = X_ct.std(axis=0)
        pct_expr = (X_ct > 0).mean(axis=0)
        p5_ct, p95_ct = np.percentile(X_ct, [5, 95], axis=0)
        del X_ct

        # ── Artefact 3: top marker genes ──────────────────────────────────────
        top_idx = np.argsort(means)[::-1][:top_n_genes]
        marker_stats = {
            gene_names[i]: {
                'mean':           float(means[i]),
                'std':            float(stds[i]),
                'pct_expressing': float(pct_expr[i]),
            }
            for i in top_idx if pct_expr[i] >= 0.10
        }
        profiles[cell_type] = {
            'n_cells':           int(len(positions)),
            'n_sampled':         int(n_sample),
            'compartment':       infer_compartment(cell_type),
            'marker_gene_stats': marker_stats,
        }

        # ── Artefact 8: CT-specific gene stats ────────────────────────────────
        # Store genes where ct_mean is meaningfully above the global mean.
        # These are the genes where CT-specific Z-scoring adds value over global Z.
        ct_stats = {}
        for i, gene in enumerate(gene_names):
            if gene not in gene_stats:
                continue
            if pct_expr[i] < CT_GENE_MIN_PCT:
                continue
            global_mean = gene_stats[gene]['mean']
            ct_mean     = float(means[i])
            if ct_mean - global_mean < CT_GENE_MIN_SPAN:
                continue
            ct_stats[gene] = {
                'mean': round(ct_mean, 4),
                'std':  round(max(float(stds[i]), 0.05), 4),  # floor prevents Z-explosion
                'p5':   round(float(p5_ct[i]), 4),
                'p95':  round(float(p95_ct[i]), 4),
                'pct_expressing': round(float(pct_expr[i]), 4),
            }
        if ct_stats:
            ct_gene_stats[cell_type] = ct_stats

        print(f'  done  {cell_type}: {len(positions):,} cells → {len(marker_stats)} markers, {len(ct_stats)} CT-enriched genes')

    return profiles, ct_gene_stats


if _OUT_PROFILES.exists() and _OUT_CT_GENES.exists():
    cell_type_profiles = load_json(_OUT_PROFILES)
    ct_gene_stats      = load_json(_OUT_CT_GENES)
    n_ct_gene_entries  = sum(len(v) for v in ct_gene_stats.values())
    print(f'Loaded CT profiles from Drive: {len(cell_type_profiles)} cell types, '
          f'{n_ct_gene_entries:,} CT-gene entries (skipped recompute)')
else:
    print('Computing per-cell-type expression profiles...')
    print('(Reads one disk slice per cell type — allow ~5 minutes)')
    t0 = time.time()
    cell_type_profiles, ct_gene_stats = compute_cell_type_profiles(
        adata, healthy_obs, gene_names, gene_stats,
        cells_per_type=CELLS_PER_TYPE, min_cells=MIN_CELLS_FOR_TYPE, top_n_genes=TOP_MARKER_GENES,
    )
    print(f'\nDone in {time.time() - t0:.1f}s. {len(cell_type_profiles)} cell types profiled.')
    n_ct_gene_entries = sum(len(v) for v in ct_gene_stats.values())
    print(f'CT-enriched gene stats: {len(ct_gene_stats)} cell types, {n_ct_gene_entries:,} gene entries')
    save_json(cell_type_profiles, _OUT_PROFILES, 'cell_type_profiles.json')
    save_json(ct_gene_stats,      _OUT_CT_GENES, 'gene_stats_by_cell_type.json')


---
## Section 8 — Pathway Baseline Activity (Artefact 4)

### What this artefact captures biologically

The lung is not an inert filter. It is a metabolically active, immunologically vigilant, continuously renewing tissue. Even in health, biological pathways run at a specific background tone:

- **Interferon signalling** sits at low but non-zero (constant antiviral surveillance)
- **TGF-β / fibrosis** is near-silent (tissue is not being actively remodelled)
- **Surfactant / AT2 function** is high (AT2 cells are continuously producing lipid-protein surfactant)
- **Mucociliary clearance** is moderate (cilia are beating and goblet cells are secreting)

This artefact records those baseline activity levels. The Abnormality Detector can then ask: **is this pathway running at its normal tone, elevated beyond it, or suppressed below it?** A pathway deviation that spans many genes is often more biologically meaningful than any single-gene deviation.

### Why mean of means?

We average the healthy expression of each pathway's trigger genes, giving equal weight to each gene. This prevents a single highly-expressed gene from dominating the pathway score. The result is a single number — the expected background level — against which a sample's pathway activity can be directly compared.

In [ ]:
# 12-pathway atlas. Each pathway has:
#   triggers   — genes whose upregulation signals pathway activation
#   suppressors — genes whose downregulation also signals activation (inverted logic)

PATHWAY_ATLAS = {
    'Interferon response': {
        'category': 'immune',
        'description': 'Antiviral/antibacterial defence programme; near-silent in healthy tissue.',
        'triggers': ['MX1', 'MX2', 'ISG15', 'ISG20', 'IFIT1', 'IFIT2', 'IFIT3',
                     'IFITM1', 'IFITM3', 'OAS1', 'OAS2', 'OAS3', 'OASL',
                     'STAT1', 'STAT2', 'IRF7', 'IRF9'],
        'suppressors': [],
    },
    'NF-kB / cytokine storm': {
        'category': 'immune',
        'description': 'Master inflammatory regulator; drives cytokine amplification loops.',
        'triggers': ['TNF', 'IL6', 'IL1B', 'CXCL8', 'CXCL10',
                     'CCL2', 'CCL3', 'CCL4', 'NFKB1', 'NFKBIA',
                     'RELA', 'IL18', 'IL12A', 'IL12B'],
        'suppressors': [],
    },
    'TGF-beta / fibrosis': {
        'category': 'remodelling',
        'description': 'Tissue remodelling and fibrosis programme; low in healthy, high in IPF/sarcoidosis.',
        'triggers': ['TGFB1', 'TGFB2', 'TGFB3', 'SMAD2', 'SMAD3',
                     'COL1A1', 'COL1A2', 'COL3A1', 'FN1', 'ACTA2',
                     'LOX', 'LOXL2', 'MMP2', 'MMP9'],
        'suppressors': [],
    },
    'Epithelial-Mesenchymal Transition (EMT)': {
        'category': 'remodelling',
        'description': 'Structural transformation where epithelial cells lose identity and gain mesenchymal traits.',
        'triggers': ['VIM', 'FN1', 'SNAI1', 'SNAI2', 'ZEB1', 'ZEB2',
                     'TWIST1', 'TWIST2', 'MMP2', 'MMP9', 'CDH2'],
        'suppressors': ['CDH1', 'EPCAM', 'OCLN', 'TJP1'],  # epithelial identity markers lost during EMT
    },
    'Surfactant / AT2 function': {
        'category': 'alveolar',
        'description': 'AT2 cell programme for producing surfactant lipids and proteins that keep alveoli open.',
        'triggers': ['SFTPC', 'SFTPB', 'SFTPA1', 'SFTPA2', 'SFTPD',
                     'ABCA3', 'LPCAT1', 'FASN', 'SLC34A2', 'NKX2-1'],
        'suppressors': [],
    },
    'Hypoxia / HIF response': {
        'category': 'metabolic',
        'description': 'Oxygen-sensing pathway; activated when lung tissue is not receiving adequate O2.',
        'triggers': ['HIF1A', 'EPAS1', 'VEGFA', 'VEGFB', 'VEGFC',
                     'LDHA', 'SLC2A1', 'BNIP3', 'BNIP3L', 'CA9',
                     'P4HA1', 'PLOD1', 'PDK1', 'PGK1'],
        'suppressors': [],
    },
    'Cell proliferation / oncogenic': {
        'category': 'oncogenic',
        'description': 'Cell cycle entry and growth control; aberrant activation underlies lung cancer.',
        'triggers': ['MKI67', 'TOP2A', 'PCNA', 'CDK1', 'CDK2', 'CDK4',
                     'CCNA2', 'CCNB1', 'CCND1', 'CCNE1', 'E2F1',
                     'MYBL2', 'BUB1', 'PLK1', 'KRAS', 'MYC'],
        'suppressors': ['RB1', 'TP53', 'CDKN1A', 'CDKN2A'],  # tumour suppressors; loss = activation
    },
    'Mucociliary clearance': {
        'category': 'airway',
        'description': 'Airway defence: ciliated cells beat, goblet cells secrete mucus to trap and clear debris.',
        'triggers': ['FOXJ1', 'DNAI1', 'DNAI2', 'DYNC2H1', 'CCNO',
                     'MUC5AC', 'MUC5B', 'MUC1', 'SCGB1A1', 'SCGB3A2'],
        'suppressors': [],
    },
    'Complement activation': {
        'category': 'immune',
        'description': 'Innate immune tagging system; marks pathogens and debris for phagocytosis.',
        'triggers': ['C1QA', 'C1QB', 'C1QC', 'C3', 'C4A', 'C4B',
                     'CFB', 'CFD', 'SERPING1', 'MBL2', 'FCN1', 'FCN2'],
        'suppressors': [],
    },
    'Vascular injury': {
        'category': 'vascular',
        'description': 'Endothelial stress and coagulation activation; marks vessel wall damage.',
        'triggers': ['ICAM1', 'VCAM1', 'SELE', 'SELP', 'VWF', 'THBD',
                     'PLAU', 'PLAUR', 'ANGPT2', 'EDN1', 'F3', 'PTGS2'],
        'suppressors': ['PECAM1', 'CDH5', 'KDR'],  # endothelial integrity markers — lost upon injury
    },
    'Myeloid activation': {
        'category': 'immune',
        'description': 'Macrophage and neutrophil activation state; elevated during infection and ARDS.',
        'triggers': ['CD14', 'CD68', 'MARCO', 'MSR1', 'MRC1', 'ITGAM',
                     'S100A8', 'S100A9', 'S100A12', 'LYZ', 'MPO', 'ELANE',
                     'CTSS', 'CTSB', 'LAMP1'],
        'suppressors': [],
    },
    'T cell exhaustion': {
        'category': 'immune',
        'description': 'Adaptive immune dysfunction; exhausted T cells cannot clear chronic infection or tumour.',
        'triggers': ['PDCD1', 'LAG3', 'HAVCR2', 'TIGIT', 'CTLA4',
                     'TOX', 'TOX2', 'BATF', 'NR4A1', 'NR4A3', 'ENTPD1'],
        'suppressors': ['TCF7', 'IL7R', 'SELL', 'CCR7'],  # naive/memory markers — lost upon exhaustion
    },
}

print(f'Pathway atlas: {len(PATHWAY_ATLAS)} pathways defined')

In [ ]:
def compute_pathway_baselines(gene_stats: dict, pathway_atlas: dict) -> dict:
    """
    Compute healthy baseline activity for each pathway.
    Uses the mean expression of pathway trigger genes that appear in gene_stats.
    Pathways with no tracked genes emit a warning.
    """
    baselines = {}

    for name, pw in pathway_atlas.items():
        tracked_triggers    = [g for g in pw['triggers']    if g in gene_stats]
        tracked_suppressors = [g for g in pw['suppressors'] if g in gene_stats]

        if not tracked_triggers:
            print(f'  WARNING: no trigger genes in gene_stats for "{name}"')
            baselines[name] = {
                'category': pw['category'], 'description': pw['description'],
                'n_trigger_genes': 0, 'n_suppressor_genes': 0,
                'trigger_mean_expr': None, 'suppressor_mean_expr': None,
                'genes_tracked': {},
            }
            continue

        trigger_means    = [gene_stats[g]['mean'] for g in tracked_triggers]
        suppressor_means = [gene_stats[g]['mean'] for g in tracked_suppressors]

        genes_tracked = {
            **{g: {'mean': gene_stats[g]['mean'], 'std': gene_stats[g]['std'], 'role': 'trigger'}
               for g in tracked_triggers},
            **{g: {'mean': gene_stats[g]['mean'], 'std': gene_stats[g]['std'], 'role': 'suppressor'}
               for g in tracked_suppressors},
        }

        baselines[name] = {
            'category':             pw['category'],
            'description':          pw['description'],
            'n_trigger_genes':      len(tracked_triggers),
            'n_suppressor_genes':   len(tracked_suppressors),
            'trigger_mean_expr':    float(np.mean(trigger_means)),
            'suppressor_mean_expr': float(np.mean(suppressor_means)) if suppressor_means else None,
            'genes_tracked':        genes_tracked,
        }

    return baselines


if _OUT_PATHWAYS.exists():
    pathway_baselines = load_json(_OUT_PATHWAYS)
    print(f'Loaded pathway_baselines from Drive: {len(pathway_baselines)} pathways (skipped recompute)')
else:
    print('Computing pathway baselines...')
    pathway_baselines = compute_pathway_baselines(gene_stats, PATHWAY_ATLAS)
    save_json(pathway_baselines, _OUT_PATHWAYS, 'pathway_baselines.json')

print('\nHealthy baseline pathway activity (log1p CP10K):')
print(f'{"Pathway":<42} {"Category":<15} {"Genes":>6} {"Trigger mean":>14}')
print('-' * 82)
for name, pw in pathway_baselines.items():
    mean_str = f'{pw["trigger_mean_expr"]:.4f}' if pw['trigger_mean_expr'] is not None else 'n/a'
    print(f'{name:<42} {pw["category"]:<15} {pw["n_trigger_genes"]:>6} {mean_str:>14}')

---
## Section 9 — Fetal Lung Reference (Artefact 5)

### Why include fetal lung data?

The fetal lung and adult healthy lung share many cell types — but the molecular programme is profoundly different. In the fetal lung:
- Cells are rapidly proliferating and differentiating
- Developmental transcription factors (*SOX9*, *ID2*, *FOXA2*) are highly active
- Surfactant production has not yet fully matured
- The extracellular matrix is being actively laid down

In a healthy adult lung, most of these fetal programmes are silenced. When they reappear in adult tissue, it signals one of a small number of events:
- **Regenerative response** to injury (AT2 cells re-entering a progenitor state to replenish damaged AT1 cells)
- **Dedifferentiation** (cancer cells reverting to an immature transcriptional state)
- **Developmental pathway hijacking** (a common oncogenic mechanism in lung adenocarcinoma)

By building a fetal reference alongside the adult baseline, the Abnormality Detector can ask: *are fetal programmes re-activated in this adult sample?* That is a specific and clinically meaningful question.

### What we compute

1. **Fetal gene statistics** — mean/std/pct_expressing for every gene in fetal lung cells
2. **Fetal-enriched genes** — genes with much higher expression in fetal vs. adult healthy lung (log fold change > 1.0)
3. **Adult-enriched genes** — genes with much higher expression in adult vs. fetal (maturation markers)
4. **Shared genes** — stable across development (housekeeping)

We do **not** mix fetal data into the adult healthy baseline. They remain separate references.

In [ ]:
if not FETAL_PATH.exists():
    print(f'WARNING: Fetal data not found at {FETAL_PATH}')
    print('Skipping fetal reference — set FETAL_PATH correctly to include it.')
    fetal_reference = {'status': 'not_computed', 'reason': 'file not found'}
else:
    print('Loading fetal lung data...')
    t0 = time.time()
    # Fetal data is 71K cells — small enough to load fully into RAM
    adata_fetal = ad.read_h5ad(FETAL_PATH)
    print(f'Loaded in {time.time() - t0:.1f}s')
    print(f'Shape: {adata_fetal.n_obs:,} cells × {adata_fetal.n_vars:,} genes')
    print(f'\nObs columns: {list(adata_fetal.obs.columns)}')
    if 'domain' in adata_fetal.obs.columns:
        print(f'\nFetal domains: {adata_fetal.obs["domain"].value_counts().to_string()}')
    elif 'cell_type' in adata_fetal.obs.columns:
        print(f'\nFetal cell types: {adata_fetal.obs["cell_type"].value_counts().head(10).to_string()}')

In [ ]:
if FETAL_PATH.exists():
    FETAL_SAMPLE_SIZE = 10_000  # sample from fetal cells for gene stats

    # Get fetal gene expression
    rng_fetal = np.random.default_rng(seed=42)
    n_fetal   = adata_fetal.n_obs
    n_fsample = min(FETAL_SAMPLE_SIZE, n_fetal)
    fetal_pos = np.sort(rng_fetal.choice(n_fetal, size=n_fsample, replace=False))

    X_fetal_raw = adata_fetal.X[fetal_pos, :]
    if issparse(X_fetal_raw):
        X_fetal_raw = X_fetal_raw.toarray()
    X_fetal_norm = normalize_cp10k_log1p(X_fetal_raw)

    # Detect Ensembl IDs in fetal dataset too
    _fetal_var0 = str(adata_fetal.var_names[0])
    if _fetal_var0.startswith('ENSG') or _fetal_var0.startswith('ENSMUSG'):
        _fsym_cols = ['feature_name', 'gene_name', 'gene_symbols', 'symbol']
        _fsym_col  = next((c for c in _fsym_cols if c in adata_fetal.var.columns), None)
        fetal_gene_names = list(adata_fetal.var[_fsym_col].astype(str)) if _fsym_col else list(adata_fetal.var_names)
        if _fsym_col:
            print(f'Fetal: using gene symbols from adata_fetal.var["{_fsym_col}"]')
        else:
            print('WARNING: fetal var_names are Ensembl IDs and no symbol column found — overlap with adult will be 0')
    else:
        fetal_gene_names = list(adata_fetal.var_names)
    fetal_pct_expr    = (X_fetal_norm > 0).mean(axis=0)
    fetal_means       = X_fetal_norm.mean(axis=0)
    fetal_stds        = X_fetal_norm.std(axis=0)

    # Build a map from gene symbol to fetal stats
    fetal_gene_stats = {
        gene: {
            'mean':           float(fetal_means[i]),
            'std':            float(fetal_stds[i]),
            'pct_expressing': float(fetal_pct_expr[i]),
        }
        for i, gene in enumerate(fetal_gene_names)
        if fetal_pct_expr[i] >= 0.01
    }

    print(f'Fetal genes tracked (≥1% expressing): {len(fetal_gene_stats):,}')

In [ ]:
if FETAL_PATH.exists():
    LOG_FC_THRESHOLD = 1.0   # log-fold-change threshold: fetal vs adult mean must differ by ≥1.0 log1p units
    PCT_THRESHOLD    = 0.10  # gene must be expressed in ≥10% of cells in the enriched group

    fetal_enriched  = {}  # high in fetal, low in adult — developmental markers
    adult_enriched  = {}  # high in adult, low in fetal — maturation markers
    shared_genes    = {}  # similar in both — housekeeping

    # Only compare genes present in both datasets
    common_genes = set(fetal_gene_stats.keys()) & set(gene_stats.keys())
    print(f'Genes shared between fetal and adult datasets: {len(common_genes):,}')

    for gene in common_genes:
        f_mean = fetal_gene_stats[gene]['mean']
        a_mean = gene_stats[gene]['mean']
        f_pct  = fetal_gene_stats[gene]['pct_expressing']
        a_pct  = gene_stats[gene]['pct_expressing']
        lfc    = f_mean - a_mean  # log-fold-change in log1p space ≈ actual fold change direction

        entry = {
            'fetal_mean':   f_mean,  'adult_mean':  a_mean,
            'fetal_pct':    f_pct,   'adult_pct':   a_pct,
            'log_fc_fetal_vs_adult': float(lfc),
        }

        if lfc >= LOG_FC_THRESHOLD and f_pct >= PCT_THRESHOLD:
            fetal_enriched[gene] = entry
        elif lfc <= -LOG_FC_THRESHOLD and a_pct >= PCT_THRESHOLD:
            adult_enriched[gene] = entry
        else:
            shared_genes[gene] = entry

    # Summarise
    print(f'\nGene classification (LFC threshold ≥ {LOG_FC_THRESHOLD}):')
    print(f'  Fetal-enriched (developmental markers): {len(fetal_enriched):,}')
    print(f'  Adult-enriched (maturation markers):    {len(adult_enriched):,}')
    print(f'  Shared / housekeeping:                  {len(shared_genes):,}')

    print(f'\nTop 15 fetal-enriched genes (genes re-activated in adult = potential dedifferentiation signal):')
    top_fetal = sorted(fetal_enriched.items(), key=lambda x: x[1]['log_fc_fetal_vs_adult'], reverse=True)[:15]
    for gene, s in top_fetal:
        print(f'  {gene:<12} LFC={s["log_fc_fetal_vs_adult"]:+.2f}  fetal={s["fetal_mean"]:.3f}  adult={s["adult_mean"]:.3f}')

    # Package the fetal reference artefact
    fetal_reference = {
        'source_file':         FETAL_PATH.name,
        'n_fetal_cells':       int(n_fetal),
        'n_sampled':           int(n_fsample),
        'n_common_genes':      int(len(common_genes)),
        'log_fc_threshold':    LOG_FC_THRESHOLD,
        'pct_threshold':       PCT_THRESHOLD,
        'fetal_enriched':      fetal_enriched,
        'adult_enriched':      adult_enriched,
        'shared_gene_count':   int(len(shared_genes)),
    }

---
## Section 10 — Spatial Visium Context (Artefact 6)

### What spatial transcriptomics adds to the model

Single-cell RNA-seq (the HLCA data) tells us *what* cells express — but it loses the information about *where in the tissue* they are. A macrophage in the bronchus and a macrophage in the alveolus can have subtly different expression programmes because they are in different microenvironments.

**Visium spatial transcriptomics** preserves spatial position. Each Visium "spot" is a ~55µm circle placed on a tissue section and captures the combined expression of all cells within that spot (~10–50 cells). Spots retain their (x, y) coordinates in the tissue.

We have two healthy lung tissue sections:
- **Bronchus** — upper airway, lined with ciliated epithelium, goblet cells, and submucosal glands
- **Parenchyma** — alveolar region, the gas-exchange interface, dominated by AT2 cells and alveolar macrophages

### What this artefact captures

For each anatomical region (bronchus, parenchyma), we compute:
- The gene expression baseline — what is normally highly expressed in this region?
- The top regionally enriched genes — genes more active in bronchus than parenchyma and vice versa
- Pathway activity per region — which pathways are more active in each anatomical context?

This gives the Abnormality Detector spatial resolution: if a patient sample shows bronchial gene programmes in what should be parenchymal tissue, that is a specific type of abnormality — potentially indicating squamous cell metaplasia or proximalisation of distal airway.

In [ ]:
def load_visium(path: Path, region_label: str) -> dict | None:
    """
    Load a Visium .h5ad file, normalize spots, and compute gene expression statistics.
    Returns None if the file does not exist.
    """
    if not path.exists():
        print(f'  WARNING: {region_label} Visium not found at {path}')
        return None

    print(f'  Loading {region_label} ({path.name})...')
    t0 = time.time()
    # Visium files are small (~4K spots) — load fully into memory
    av = ad.read_h5ad(path)
    print(f'    {av.n_obs:,} spots × {av.n_vars:,} genes ({time.time() - t0:.1f}s)')

    X_raw = av.X
    if issparse(X_raw):
        X_raw = X_raw.toarray()

    X_norm = normalize_cp10k_log1p(X_raw)
    # Detect Ensembl IDs in Visium data
    _vis_var0 = str(av.var_names[0])
    if _vis_var0.startswith('ENSG') or _vis_var0.startswith('ENSMUSG'):
        _vsym_cols = ['feature_name', 'gene_name', 'gene_symbols', 'symbol']
        _vsym_col  = next((c for c in _vsym_cols if c in av.var.columns), None)
        visium_gene_names = list(av.var[_vsym_col].astype(str)) if _vsym_col else list(av.var_names)
        if not _vsym_col:
            print(f'  WARNING: {region_label} Visium var_names are Ensembl IDs and no symbol column found')
    else:
        visium_gene_names = list(av.var_names)

    pct_expr = (X_norm > 0).mean(axis=0)
    keep     = pct_expr >= 0.01
    kept_genes = [g for g, k in zip(visium_gene_names, keep) if k]

    X_kept   = X_norm[:, keep]
    means    = X_kept.mean(axis=0)
    stds     = X_kept.std(axis=0)
    p5, p95  = np.percentile(X_kept, [5, 95], axis=0)

    # Top 200 most expressed genes — the spatial 'signature' of this region
    top_idx  = np.argsort(means)[::-1][:200]

    gene_stats_region = {
        kept_genes[i]: {
            'mean':           float(means[i]),
            'std':            float(stds[i]),
            'p5':             float(p5[i]),
            'p95':            float(p95[i]),
            'pct_expressing': float(pct_expr[keep][i]),
        }
        for i in top_idx
    }

    # Pathway activity in this region
    region_pathway_activity = {}
    for pname, pw in PATHWAY_ATLAS.items():
        # Build a quick lookup
        gene_to_idx = {g: i for i, g in enumerate(kept_genes)}
        tracked_in_region = [g for g in pw['triggers'] if g in gene_to_idx]
        if tracked_in_region:
            region_means = [float(means[gene_to_idx[g]]) for g in tracked_in_region]
            region_pathway_activity[pname] = {
                'n_genes_tracked':  len(tracked_in_region),
                'trigger_mean_expr': float(np.mean(region_means)),
            }

    return {
        'region':              region_label,
        'source_file':         path.name,
        'n_spots':             int(av.n_obs),
        'n_genes_tracked':     len(kept_genes),
        'top_expressed_genes': gene_stats_region,
        'pathway_activity':    region_pathway_activity,
    }


print('Loading Visium datasets...')
bronchus_baseline   = load_visium(VISIUM_BRONCHUS_PATH,   'bronchus')
parenchyma_baseline = load_visium(VISIUM_PARENCHYMA_PATH, 'parenchyma')

In [ ]:
def compute_spatial_enrichment(bronchus: dict, parenchyma: dict) -> dict:
    """
    Find genes enriched in bronchus vs. parenchyma and vice versa.
    Enrichment = genes that appear in one region's top-200 but not the other,
    or that differ substantially in mean expression.
    """
    b_genes = set(bronchus['top_expressed_genes'].keys())
    p_genes = set(parenchyma['top_expressed_genes'].keys())

    bronchus_specific  = sorted(b_genes - p_genes)  # in top-200 bronchus only
    parenchyma_specific = sorted(p_genes - b_genes)  # in top-200 parenchyma only
    shared = sorted(b_genes & p_genes)

    return {
        'bronchus_enriched':   bronchus_specific[:50],   # store top 50
        'parenchyma_enriched': parenchyma_specific[:50],
        'shared_top_genes':    shared[:50],
    }


if bronchus_baseline and parenchyma_baseline:
    enrichment = compute_spatial_enrichment(bronchus_baseline, parenchyma_baseline)

    print('Bronchus-enriched genes (top-50, airway signature):')
    print('  ' + ', '.join(enrichment['bronchus_enriched'][:20]))
    print('\nParenchyma-enriched genes (top-50, alveolar signature):')
    print('  ' + ', '.join(enrichment['parenchyma_enriched'][:20]))

    print('\nPathway activity comparison (bronchus vs. parenchyma):')
    b_pw = bronchus_baseline.get('pathway_activity', {})
    p_pw = parenchyma_baseline.get('pathway_activity', {})
    common_pw = set(b_pw) & set(p_pw)
    print(f'{"Pathway":<42} {"Bronchus":>12} {"Parenchyma":>12} {"Difference":>12}')
    print('-' * 80)
    for pw_name in sorted(common_pw):
        b_val = b_pw[pw_name]['trigger_mean_expr']
        p_val = p_pw[pw_name]['trigger_mean_expr']
        diff  = b_val - p_val
        print(f'{pw_name:<42} {b_val:>12.4f} {p_val:>12.4f} {diff:>+12.4f}')

    spatial_baselines = {
        'bronchus':   bronchus_baseline,
        'parenchyma': parenchyma_baseline,
        'enrichment': enrichment,
    }
else:
    print('One or both Visium files missing — spatial_baselines will be partial.')
    spatial_baselines = {
        'bronchus':   bronchus_baseline or {'status': 'not_computed'},
        'parenchyma': parenchyma_baseline or {'status': 'not_computed'},
    }

---
## Section 11 — Save All Artefacts

Seven JSON files, all to `models/respiratory_model/`. The total size will be a few MB — small enough to version, fast to load. This is the compression from 20+ GB of raw data into structured knowledge.

In [ ]:
def save_json(data: dict, path: Path, label: str) -> None:
    with open(path, 'w') as f:
        json.dump(data, f, indent=2, allow_nan=False)
    kb = path.stat().st_size / 1024
    print(f'  {"✓":<3} {label:<40} → {path.name}  ({kb:.0f} KB)')


meta = {
    'built_at':             datetime.now(timezone.utc).isoformat(),
    'n_cells':              int(n_healthy),
    'n_donors':             int(n_donors),
    'n_genes':              int(len(gene_stats)),
    'n_cell_types':         int(len(cell_type_profiles)),
    'n_pathways':           int(len(pathway_baselines)),
    'n_ct_gene_entries':    int(n_ct_gene_entries),
    'source_file':          HLCA_FULL_PATH.name,
    'fetal_source':         FETAL_PATH.name if FETAL_PATH.exists() else None,
    'visium_bronchus':      VISIUM_BRONCHUS_PATH.name if VISIUM_BRONCHUS_PATH.exists() else None,
    'visium_parenchyma':    VISIUM_PARENCHYMA_PATH.name if VISIUM_PARENCHYMA_PATH.exists() else None,
    'disease_filter':       'disease == normal',
    'normalization':        'log1p(CP10K)',
    'cell_type_column':     'ann_finest_level',
    'global_sample_size':   GLOBAL_SAMPLE_SIZE,
    'cells_per_type':       CELLS_PER_TYPE,
    'min_cells_for_type':   MIN_CELLS_FOR_TYPE,
    'min_expressing_frac':  MIN_EXPRESSING_FRAC,
}

print(f'Saving to: {MODEL_OUT_DIR}\n')
save_json(meta,               MODEL_OUT_DIR / 'meta.json',                       'Build metadata')
save_json(composition,        MODEL_OUT_DIR / 'cell_type_composition.json',      'Cell type composition')
save_json(gene_stats,         MODEL_OUT_DIR / 'gene_stats.json',                 'Global gene statistics')
save_json(cell_type_profiles, MODEL_OUT_DIR / 'cell_type_profiles.json',         'Cell type profiles')
save_json(pathway_baselines,  MODEL_OUT_DIR / 'pathway_baselines.json',          'Pathway baselines')
save_json(fetal_reference,    MODEL_OUT_DIR / 'fetal_reference.json',            'Fetal lung reference')
save_json(spatial_baselines,  MODEL_OUT_DIR / 'spatial_baselines.json',          'Spatial Visium baselines')
save_json(ct_gene_stats,      MODEL_OUT_DIR / 'gene_stats_by_cell_type.json',    'CT-specific gene stats')

print('\nAll artefacts saved.')


In [ ]:
expected = [
    ('meta.json',                        'Build metadata + parameters'),
    ('cell_type_composition.json',       'Cellular architecture of healthy lung'),
    ('gene_stats.json',                  'Full gene expression distribution in healthy tissue'),
    ('cell_type_profiles.json',          'Molecular identity of each cell type'),
    ('pathway_baselines.json',           'Background activity of 12 biological pathways'),
    ('fetal_reference.json',             'Developmental vs. mature gene signatures'),
    ('spatial_baselines.json',           'Region-specific expression (bronchus / parenchyma)'),
    ('gene_stats_by_cell_type.json',     'CT-specific gene stats for suppression detection'),
]

total_kb = 0
print(f'{"File":<45} {"Size":>8}  Description')
print('-' * 90)
for fname, desc in expected:
    fpath = MODEL_OUT_DIR / fname
    if fpath.exists():
        kb = fpath.stat().st_size / 1024
        total_kb += kb
        print(f'{fname:<45} {kb:>7.0f}K  {desc}')
    else:
        print(f'{fname:<45} MISSING   {desc}')

print(f'\nTotal: {total_kb:.0f} KB ({total_kb / 1024:.1f} MB)')


---
## Section 12 — Validation and Sanity Checks

A healthy baseline that contains errors will silently corrupt every downstream analysis. We run targeted checks against known biological facts — things we know to be true about the healthy human lung.

In [ ]:
errors = []
warnings_list = []

# ── 1. Cell type fractions must sum to ≈1.0 ──────────────────────────────────
frac_sum = sum(v['mean_fraction'] for v in composition.values())
if abs(frac_sum - 1.0) > 0.05:
    errors.append(f'Composition fractions sum to {frac_sum:.4f}, expected ≈1.0')
else:
    print(f'[OK] Composition fractions sum: {frac_sum:.4f}')

# ── 2. Surfactant genes present and at reasonable levels ─────────────────────
# SFTPC is the canonical AT2 marker — if absent, something went wrong with gene names
for gene, expected_range in [('SFTPC', (0.1, 5.0)), ('SFTPB', (0.05, 4.0))]:
    if gene not in gene_stats:
        warnings_list.append(f'{gene} absent from gene_stats — check gene name format')
    else:
        m = gene_stats[gene]['mean']
        lo, hi = expected_range
        if not (lo <= m <= hi):
            warnings_list.append(f'{gene} mean={m:.3f} outside expected range [{lo}, {hi}]')
        else:
            print(f'[OK] {gene} mean={m:.3f}')

# ── 3. Interferon genes should be low in healthy tissue ───────────────────────
# MX1 is near-silent in healthy lung (activated only during viral infection)
if 'MX1' in gene_stats:
    mx1_mean = gene_stats['MX1']['mean']
    if mx1_mean > 1.5:
        warnings_list.append(f'MX1 mean={mx1_mean:.3f} — unexpectedly high for healthy tissue (expected <1.5)')
    else:
        print(f'[OK] MX1 (interferon marker) mean={mx1_mean:.3f} — appropriately low in healthy')

# ── 4. Major cell types present in composition ────────────────────────────────
for term in ['macrophage', 'AT2', 'fibroblast', 'ciliated', 'endothelial']:
    found = [ct for ct in composition if term.lower() in ct.lower()]
    if found:
        frac = composition[found[0]]['mean_fraction']
        print(f'[OK] "{term}" present as "{found[0]}" (fraction={frac:.3%})')
    else:
        warnings_list.append(f'No cell type matching "{term}" — check ann_finest_level values in HLCA')

# ── 5. Gene expression values within physiological range ─────────────────────
all_means = [v['mean'] for v in gene_stats.values()]
max_mean  = max(all_means)
if max_mean > 15:
    warnings_list.append(f'Max gene mean {max_mean:.2f} > 15 — unusually high for log1p CP10K')
else:
    print(f'[OK] Gene expression range: {min(all_means):.3f} – {max_mean:.3f} (log1p CP10K)')

# ── 6. Fetal reference structure check ───────────────────────────────────────
if 'fetal_enriched' in fetal_reference:
    n_fe = len(fetal_reference['fetal_enriched'])
    n_ae = len(fetal_reference['adult_enriched'])
    if n_fe < 10:
        warnings_list.append(f'Only {n_fe} fetal-enriched genes — may indicate data or threshold issue')
    else:
        print(f'[OK] Fetal reference: {n_fe} fetal-enriched, {n_ae} adult-enriched genes')

# ── 7. Spatial baselines loaded ───────────────────────────────────────────────
for region in ['bronchus', 'parenchyma']:
    entry = spatial_baselines.get(region, {})
    if entry.get('status') == 'not_computed':
        warnings_list.append(f'Spatial baseline for {region} not computed (file missing)')
    elif 'n_spots' in entry:
        print(f'[OK] Spatial {region}: {entry["n_spots"]} spots, {entry["n_genes_tracked"]} genes')

# ── 8. All artefact files loadable ───────────────────────────────────────────
loaded_meta = json.loads((MODEL_OUT_DIR / 'meta.json').read_text())
assert loaded_meta['n_genes'] > 0
print(f'[OK] meta.json round-trip: {loaded_meta["n_cells"]:,} cells, {loaded_meta["n_genes"]:,} genes')

# ── Summary ───────────────────────────────────────────────────────────────────
print()
for e in errors:        print(f'[ERROR] {e}')
for w in warnings_list: print(f'[WARN]  {w}')
if not errors:
    print('Validation passed.' + (' (with warnings)' if warnings_list else ' All checks OK.'))

---
## Section 13 — Model Summary

In [ ]:
print('=' * 65)
print('  HEALTHY RESPIRATORY MODEL — BUILD SUMMARY')
print('=' * 65)
print(f'  Built at:         {meta["built_at"]}')
print(f'  Primary source:   {meta["source_file"]}')
print(f'  Fetal source:     {meta["fetal_source"] or "not included"}')
print(f'  Visium bronchus:  {meta["visium_bronchus"] or "not included"}')
print(f'  Visium parenchyma:{meta["visium_parenchyma"] or "not included"}')
print()
print(f'  Healthy cells:    {meta["n_cells"]:,}')
print(f'  Healthy donors:   {meta["n_donors"]}')
print(f'  Genes tracked:    {meta["n_genes"]:,}')
print(f'  Cell types:       {meta["n_cell_types"]}')
print(f'  Pathways:         {meta["n_pathways"]}')
print()
print('  Top 10 cell types by abundance in healthy lung:')
sorted_types = sorted(composition.items(), key=lambda x: x[1]['mean_fraction'], reverse=True)
for ct, s in sorted_types[:10]:
    bar = '█' * int(s['mean_fraction'] * 200)
    print(f'    {ct:<42} {s["mean_fraction"]:>6.2%}  {bar}')
print()
print(f'  Artefacts saved to: {MODEL_OUT_DIR}')
print('=' * 65)

---
## Section 14 — Next Steps: Building the Abnormality Detector

This notebook produced a seven-file portrait of the healthy human respiratory system. The next notebook (`03_abnormality_detector.ipynb`) will build the system that uses this portrait to characterise how a new sample deviates from it.

Here is the proposed architecture for the Abnormality Detector — grounded in what we now have.

---

### Input

A dictionary of `{gene_symbol: expression_value}` — the log1p CP10K expression profile of a new sample (bulk RNA-seq, pseudobulk from scRNA-seq, or a single-cell profile). Minimum 50 genes required.

---

### Dimension 1: Cell Type Composition Deviation

**Available artefact**: `cell_type_composition.json`

**Method**: If the sample has cell type annotation (e.g., deconvolution results or from scRNA-seq clustering), compare the sample's cell type fractions to the healthy mean fractions.

For each cell type:
```
z_score = (sample_fraction - healthy_mean_fraction) / (healthy_std_fraction + ε)
```

A Z-score > +2 means this cell type is expanded beyond healthy range. A Z-score < -2 means it is depleted. The **direction** and **magnitude** of composition shift are the primary output.

**Example**: `alveolar macrophage Z = +3.2` → expanded macrophage population, suggesting active inflammation or infection.

---

### Dimension 2: Gene-Level Deviation

**Available artefact**: `gene_stats.json`

**Method**: For each gene in the sample that is also in `gene_stats`:
```
z_score = (sample_value - healthy_mean) / (healthy_std + ε)
```

Additionally, compute the **percentile rank** of the sample value within the healthy distribution:
- If `sample_value > p95`: the gene is in the top 5% of healthy — borderline high
- If `sample_value > (p95 + 2*std)`: definitively outside healthy range

Report only genes with |Z| > 2.0 — the rest are within the normal range. Rank by magnitude.

**Example**: `MX1 Z = +6.1, above p99` → massive interferon activation, consistent with active viral infection.

---

### Dimension 3: Pathway Deviation

**Available artefact**: `pathway_baselines.json`

**Method**: For each pathway, compute the sample's mean expression of that pathway's trigger genes. Compare to the healthy baseline:
```
sample_activity = mean(sample[g] for g in pathway_triggers if g in sample)
deviation = sample_activity - baseline_trigger_mean_expr
```

A deviation > 1.0 log1p unit (roughly 2.7-fold) in trigger expression is clinically meaningful. For pathways with suppressors, check if suppressor expression has dropped (a double signal of activation).

**Example**: `TGF-β / fibrosis: +1.8 log1p above baseline, suppressor genes −0.6` → strong fibrotic signalling active.

---

### Dimension 4: Cell Type Functional Deviation

**Available artefact**: `cell_type_profiles.json`

**Method**: If the sample's cell type is known (or estimated), compare its expression of the cell type's marker genes against the per-cell-type healthy profile.

For each marker gene of cell type C:
```
z_score = (sample_value - profile_mean[C][gene]) / (profile_std[C][gene] + ε)
```

A cell type that is present in the right proportion (Dimension 1 is normal) but whose marker genes are not expressing correctly (this dimension is abnormal) is **functionally impaired**. This is a subtle but important distinction.

**Example**: AT2 cells present at normal fraction, but `SFTPC Z = −3.1` → AT2 cells are present but not producing surfactant normally. Consistent with early fibrosis or AT2 dysfunction.

---

### Dimension 5: Developmental Deviation

**Available artefact**: `fetal_reference.json`

**Method**: Compute a **fetal reactivation score** — the mean expression of fetal-enriched genes in the sample, normalised by adult-enriched gene expression.

A high score means the sample is expressing programmes that belong to fetal development, not adult health. This is a specific signal for:
- Dedifferentiation (common in lung adenocarcinoma)
- AT2-to-AT1 transitional cell states (post-injury regeneration)
- Abnormal progenitor expansion

---

### Dimension 6: Spatial Context Mismatch

**Available artefact**: `spatial_baselines.json`

**Method** (for samples with known anatomical origin): compare the sample's expression profile against the expected profile for its tissue region (bronchus or parenchyma). A sample from a parenchymal biopsy that strongly expresses bronchial markers suggests **proximalisation** of the distal airway — a known finding in COPD and interstitial lung disease.

Compare sample expression to:
```
bronchial_score   = mean(sample[g] for g in bronchus_enriched_genes)
parenchymal_score = mean(sample[g] for g in parenchyma_enriched_genes)
```

If the sample's anatomical origin is parenchyma but `bronchial_score >> parenchymal_score`, flag spatial identity mismatch.

---

### Overall Deviation Score

A composite score across dimensions, weighted by biological confidence:
```
overall = (
    0.30 × composition_score      +  # cell type proportions
    0.25 × gene_deviation_score   +  # individual gene Z-scores
    0.20 × pathway_score          +  # coordinated process deviations
    0.15 × functional_score       +  # per-cell-type marker expression
    0.10 × developmental_score       # fetal reactivation
)
```

This score is a **biological deviation index**, not a disease probability. It tells a clinician: *how far is this sample from healthy, and in which dimensions?* The clinical interpretation remains with the expert.

---

### What the Abnormality Detector does NOT do

- It does not name diseases
- It does not recommend treatments
- It does not replace clinical judgment

It describes the biology. The clinician in Layer 3 names the pathology.

---

### Implementation target

The Abnormality Detector will live in `app/services/anomaly_detector.py`, which already exists as a skeleton. Notebook 03 will implement and validate each dimension above, using test inputs from known HLCA disease cells — not to train on disease, but to confirm the detector produces biologically coherent deviation profiles when given a sample that is known to be abnormal.